In [1]:
!pip install transformers torchaudio datasets librosa soundfile jiwer
!pip install pydub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yasiashpot/librispeech")

print("Path to dataset files:", path)

100%|██████████| 3.34G/3.34G [02:36<00:00, 22.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/yasiashpot/librispeech/versions/1


In [4]:
import torch
import torchaudio
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from Ifgsm import iterative_untargeted_fgsm_attack_single_audio
from fgsm import fgsm_attack_single_audio
from utils import compute_transcription, preprocess_audio

In [5]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder)

In [6]:
file_path = "/root/.cache/kagglehub/datasets/yasiashpot/librispeech/versions/1/LibriSpeech/dev-clean/174/168635/174-168635-0000.wav"

In [8]:
from IPython.display import Audio
Audio(filename=file_path)

In [13]:

def compute_transcription(input_data, model, processor, device):
    """
    Compute the transcription for a given audio file or waveform using the specified model and processor.

    Args:
        input_data (str or torch.Tensor): Path to the audio file or preprocessed waveform.
        model (torch.nn.Module): Pre-trained Wav2Vec2ForCTC model.
        processor (Wav2Vec2Processor): Processor to handle audio and text.
        device (str): Device to perform computation on ('cpu' or 'cuda').

    Returns:
        str: Transcription of the input audio or waveform.
    """
    if isinstance(input_data, str):  # If input_data is a file path
        waveform = preprocess_audio(input_data)
    elif isinstance(input_data, torch.Tensor):  # If input_data is already a waveform
        waveform = input_data
    else:
        raise ValueError("input_data must be a file path (str) or a waveform (torch.Tensor)")

    inputs = processor(waveform.numpy(), sampling_rate=16000, return_tensors="pt", padding=True)
    inputs = {key: val.to(device) for key, val in inputs.items()}  # ensure inputs are on the specified device

    with torch.no_grad():
        logits = model(inputs["input_values"]).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.decode(predicted_ids[0])
    return transcription


In [14]:
waveform = preprocess_audio(file_path)
orig_transcription = compute_transcription(file_path, model, processor, device)
print("Original Transcription:", orig_transcription)

target_text = orig_transcription
epsilon = 0.2

adv_waveform = fgsm_attack_single_audio(model, processor, waveform, target_text, epsilon)
adv_transcription = compute_transcription(adv_waveform, model, processor, device)
print("Adversarial Transcription:", adv_transcription)

Original Transcription: HE HAD NEVER BEEN FATHER LOVER HUSBAND FRIEND
Adversarial Transcription: HE HAD NEVER BEEN FATHER LOTHER HUSBAND HAD


In [15]:
Audio(adv_waveform.numpy(), rate=16000)


In [16]:
file_path = "/root/.cache/kagglehub/datasets/yasiashpot/librispeech/versions/1/LibriSpeech/dev-clean/174/168635/174-168635-0010.wav"
waveform = preprocess_audio(file_path)

orig_transcription = compute_transcription(waveform, model, processor, device)
print("Original Transcription:", orig_transcription)

Original Transcription: WHEN THESE TWO SOULS PERCEIVED EACH OTHER THEY RECOGNIZED EACH OTHER AS NECESSARY TO EACH OTHER AND EMBRACED EACH OTHER CLOSELY


In [17]:
Audio(waveform.numpy(), rate=16000)


In [18]:
true_text = orig_transcription  # Replace with the true transcription
epsilon = 0.1  # Maximum allowed perturbation
num_steps = 100  # Number of iterations

adv_waveform = iterative_untargeted_fgsm_attack_single_audio(model, processor, waveform, true_text, epsilon, num_steps)
adv_transcription = compute_transcription(adv_waveform, model, processor, device)
print("Original    Transcription:", orig_transcription)
print("Adversarial Transcription:", adv_transcription)

Original    Transcription: WHEN THESE TWO SOULS PERCEIVED EACH OTHER THEY RECOGNIZED EACH OTHER AS NECESSARY TO EACH OTHER AND EMBRACED EACH OTHER CLOSELY
Adversarial Transcription: WHEN THESE TWO SOULS FOR SAFETY CHUTHAR THEY RECOGNIZED TO CHUTHARK AS NECESSARY TO READ CHUTHARK AND EMBRACED EACH OTHER CLOSELY


In [19]:
from IPython.display import Audio
Audio(adv_waveform.numpy(), rate=16000)
